In [27]:
import anndata as ad
import pandas as pd
import anndata as ad
import scanpy as sc
import os
import random

In [28]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [29]:
import sys
# sys.path.insert(0, '/home/jupyter/scFM_eval_')
sys.path.insert(0, '..')
from evaluation.eval import EmbeddingEvaluator

In [30]:
from os.path import join

In [31]:
base_dir = '/home/haitham/mnt/__output_v3/exp'


search_text = 'cell_type'
folders= [] 
# Use os.walk to search through all subdirectories
for root, dirs, files in os.walk(base_dir):
    for directory in dirs:
        # print(directory)
        if search_text in directory:
            # Print the full path of matching folders
            print(os.path.join(root, directory))
            folders.append(os.path.join(root, directory))

/home/haitham/mnt/__output_v3/exp/hvg/seurat_4096/brca_cell_type_hvg_seurat_4096_brca_cell_type
/home/haitham/mnt/__output_v3/exp/state/se600m_epoch16/brca_cell_type_state_se600m_epoch16_brca_cell_type
/home/haitham/mnt/__output_v3/exp/scfoundation/brca_cancer_cells/brca_cell_type_scfoundation_brca_cancer_cells_brca_cell_type
/home/haitham/mnt/__output_v3/exp/geneformer/V2-104M_CLcancer-i4096/brca_cell_type_geneformer_V2-104M_CLcancer-i4096_brca_cell_type
/home/haitham/mnt/__output_v3/exp/geneformer/V2-316M-i4096/brca_cell_type_geneformer_V2-316M-i4096_brca_cell_type
/home/haitham/mnt/__output_v3/exp/geneformer/V1-10M-i2048/brca_cell_type_geneformer_V1-10M-i2048_brca_cell_type
/home/haitham/mnt/__output_v3/exp/geneformer/V2-104M-i4096/brca_cell_type_geneformer_V2-104M-i4096_brca_cell_type
/home/haitham/mnt/__output_v3/exp/scgpt/cancer-i2048/brca_cell_type_scgpt_cancer-i2048_brca_cell_type
/home/haitham/mnt/__output_v3/exp/scgpt/human-i2048/brca_cell_type_scgpt_human-i2048_brca_cell_typ

In [32]:
# base_dir = '/home/jupyter/mnt/__output_clean/brca_full/cell_type'
# base_dir = '/home/jupyter/mnt/__output_clean/brca_full/cell_type'

# gf_model_files = [
#     'hvg',
#     'pca',
#     # 'scgpt',
#     # 'scgpt_cancer',
#     'scvi_donor_id',
#     'Geneformer-V2-104M',
#                   'Geneformer-V2-104M_CLcancer',
#                   # 'Geneformer-V2-104M_continue',
#                   'Geneformer-V2-316M',
#                   'gf-6L-30M-i2048',
#                   'gf-6L-30M-i2048_continue'
#                  ]



In [33]:
base_dir

'/home/haitham/mnt/__output_v3/exp'

In [34]:
# subfolders =[entry.path for entry in os.scandir(base_dir) if entry.is_dir()]
# subfolders

In [35]:
def get_embd_key(model_name):

    if 'pca' in model_name :
        key = f'X_pca'    
        
    if 'hvg' in model_name :
        key = f'X_hvg'    
        
    if 'scvi' in model_name :
        key = f'X_scVI'
    if 'scgpt' in model_name :
        key = f'X_scGPT'
    if 'Geneformer' in model_name :
        key = f'X_geneformer'
    if 'gf' in model_name :
        key = f'X_geneformer'
        
    if 'scfoundation' in model_name:
        key = f'X_scfoundation'
    
    if 'cellplm' in model_name:
        key = f'X_CellPLM'
    
    if 'scimilarity' in model_name:
        key = f'X_scimilarity'
    return key

In [52]:
f

'/home/haitham/mnt/__output_v3/exp/hvg/seurat_4096/brca_cell_type_hvg_seurat_4096_brca_cell_type'

In [36]:
f = folders[0]
embedding_key = get_embd_key(f)
embedding_key

'X_hvg'

In [37]:
# f= gf_model_files[0]
# embedding_key = get_embd_key(f)
# embedding_key

In [38]:
# fname= join(base_dir, f)
# data_fname= join(fname,'data.h5ad' )
# embs = ad.read_h5ad(data_fname)

In [39]:
data_fname= join(f,'data.h5ad' )
embs = ad.read_h5ad(data_fname)

In [40]:
embs.shape

(41101, 3784)

In [21]:
embs.obs.timepoint.value_counts()

timepoint
Pre    41101
Name: count, dtype: int64

In [12]:
# data_fname= join('/home/jupyter/mnt/__output_clean/brca_full/cell_type/scvi_donor_id','data.h5ad' )
# embs = ad.read_h5ad(data_fname)

In [49]:
embs

AnnData object with n_obs × n_vars = 41101 × 3784
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'cell_id', 'donor_id', 'timepoint', 'outcome', 'Cancer_type', 'cell_types', 'cohort', 'pre_post', 'donor_id_pre_post', 'donor_id_outcome', 'donor_id_cell_types', 'donor_id_cell_types_pre_post', 'sample_id_pre_post_outcome', 'enough_cells', 'Study_name', 'Primary_or_met', 'RNA_snn_res.0.8', 'seurat_clusters', 'ident', 'n_genes_by_counts', 'total_counts', 'n_genes', 'label', 'batch'
    obsm: 'X_hvg'

In [50]:
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection


In [51]:
bm = Benchmarker(
    embs,
    batch_key="donor_id",
    label_key="cell_types",
    bio_conservation_metrics=BioConservation(),
    batch_correction_metrics=BatchCorrection(),
    embedding_obsm_keys=["X_hvg"],
    n_jobs=6,
)
bm.benchmark()

Embeddings:   0%|          | 0/1 [00:01<?, ?it/s]io conservation: isolated_labels]


TypeError: Argument '<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 21546 stored elements and shape (256, 3784)>
  Coords	Values
  (0, 46)	3.898291552055616
  (0, 87)	3.898291552055616
  (0, 118)	4.460644538045145
  (0, 252)	5.147471546815412
  (0, 320)	3.898291552055616
  (0, 374)	3.898291552055616
  (0, 391)	3.898291552055616
  (0, 393)	3.898291552055616
  (0, 396)	5.462494840334242
  (0, 398)	4.460644538045145
  (0, 419)	4.273368883688006
  (0, 420)	3.898291552055616
  (0, 422)	3.898291552055616
  (0, 445)	3.898291552055616
  (0, 447)	4.273368883688006
  (0, 478)	4.273368883688006
  (0, 484)	3.898291552055616
  (0, 486)	3.898291552055616
  (0, 526)	3.898291552055616
  (0, 527)	3.898291552055616
  (0, 531)	3.898291552055616
  (0, 541)	4.667155363342885
  (0, 558)	5.231243936324771
  (0, 560)	4.580687557874254
  (0, 565)	3.898291552055616
  :	:
  (255, 1053)	4.767386831201412
  (255, 1055)	6.664467363179806
  (255, 1056)	4.767386831201412
  (255, 1068)	4.767386831201412

In [41]:
subsampled_adata = sc.pp.subsample(
        embs, n_obs=min(10_000, embs.n_obs), copy=True, random_state=i
    )

In [48]:
subsampled_adata

AnnData object with n_obs × n_vars = 10000 × 3784
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'cell_id', 'donor_id', 'timepoint', 'outcome', 'Cancer_type', 'cell_types', 'cohort', 'pre_post', 'donor_id_pre_post', 'donor_id_outcome', 'donor_id_cell_types', 'donor_id_cell_types_pre_post', 'sample_id_pre_post_outcome', 'enough_cells', 'Study_name', 'Primary_or_met', 'RNA_snn_res.0.8', 'seurat_clusters', 'ident', 'n_genes_by_counts', 'total_counts', 'n_genes', 'label', 'batch'
    obsm: 'X_hvg'

In [47]:
subsampled_adata.uns['neighbors']

KeyError: 'neighbors'

In [44]:
subsampled_adata.obsp['distances']

KeyError: 'distances'

In [26]:
for i in range(1):
    subsampled_adata = sc.pp.subsample(
        embs, n_obs=min(10_000, embs.n_obs), copy=True, random_state=i
    )
    for k in ("distances", "connectivities"):
        subsampled_adata.obsp.pop(k, None)
    subsampled_adata.uns.pop("neighbors", None)
    # ensure batch/label columns match EmbeddingEvaluator + are plain strings
    # subsampled_adata.obs["batch"] = ...
    # subsampled_adata.obs["label"] = ...
    ev = EmbeddingEvaluator(
        subsampled_adata, embedding_key="X_hvg", save_dir=".", auto_subsample=False
    )
    ret_dict = ev.evaluate()

/home/haitham/scFM_eval/.pixi/envs/geneformer/lib/python3.11/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(labels)
/home/haitham/scFM_eval/.pixi/envs/geneformer/lib/python3.11/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(labels)
/home/haitham/scFM_eval/.pixi/envs/geneformer/lib/python3.11/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(labels)
/home/haitham/scFM_eval/.pixi/envs/geneformer/lib/python3.11/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated 

In [25]:
# for i in range(10):
#     subsampled_adata = sc.pp.subsample(embs, n_obs=10000, copy=True, random_state=None) 
#     subsampled_adata.obsp.pop('distances', None)
#     subsampled_adata.obsp.pop('connectivities', None)
#     subsampled_adata.uns.pop("neighbors", None)
#     ev = EmbeddingEvaluator( subsampled_adata, embedding_key='X_hvg', save_dir='.', auto_subsample=False)
#     ret_dict = ev.evaluate()
    # mid_results.append(ret_dict)

/home/haitham/scFM_eval/.pixi/envs/geneformer/lib/python3.11/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(labels)
/home/haitham/scFM_eval/.pixi/envs/geneformer/lib/python3.11/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(labels)
/home/haitham/scFM_eval/.pixi/envs/geneformer/lib/python3.11/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(labels)
/home/haitham/scFM_eval/.pixi/envs/geneformer/lib/python3.11/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated 

SystemError: CPUDispatcher(<function nn_descent at 0x76fbbf7f0f40>) returned a result with an exception set

In [ ]:
ret_dict

In [14]:
ret_dict

{'NMI_cluster/label': 0.7343039292254105,
 'ARI_cluster/label': 0.6360053569781767,
 'ASW_label': 0.5950561538338661,
 'graph_conn': 0.9944108719425653,
 'ASW_batch': 0.4788084588944912,
 'ASW_label/batch': 0.79703135790906,
 'PCR_batch': 0.17971466604208303,
 'avg_bio': 0.6551218133458178}

In [40]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [9]:
embsa

AnnData object with n_obs × n_vars = 41101 × 100
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'cell_id', 'donor_id', 'timepoint', 'outcome', 'Cancer_type', 'cell_types', 'cohort', 'pre_post', 'donor_id_pre_post', 'donor_id_outcome', 'donor_id_cell_types', 'donor_id_cell_types_pre_post', 'sample_id_pre_post_outcome', 'enough_cells', 'Study_name', 'Primary_or_met', 'RNA_snn_res.0.8', 'seurat_clusters', 'ident', 'n_genes_by_counts', 'total_counts', 'n_genes', 'label', 'batch', '_scvi_batch', '_scvi_labels'
    obsm: 'X_scVI'

In [41]:
results_mean= []
results_std= []
all_results= {}
seed = random.randint(0, 100000)
# seed = 42
for i, fname in enumerate(subfolders):
    
    model_name = os.path.basename(fname)
    print(i, model_name)
    embedding_key = get_embd_key(fname)
    print( f'    {embedding_key}')

    try: 
        data_fname= join(fname,'data.h5ad' )
        embs = ad.read_h5ad(data_fname)
        embs_ = embs.obsm[embedding_key]
        # print(embs.obsm)
    except:
        print(f'-------- Error: cannot open {model_name}------------')
        continue 
        
#     print(data_fname)
    
    
    mid_results=[]
    for i in range(10):
        subsampled_adata = sc.pp.subsample(embs, n_obs=10000, copy=True, random_state=None) 
        ev = EmbeddingEvaluator( subsampled_adata, embedding_key=embedding_key, save_dir='.', auto_subsample=False)
        ret_dict = ev.evaluate()
        mid_results.append(ret_dict)
        # print(ret_dict)
    
    df = pd.DataFrame(mid_results)
    dd = df.mean()
    dd['model'] = model_name
    results_mean.append(dd)

    dd = df.std()
    dd['model'] = model_name
    results_std.append(dd)
    
    all_results[model_name] = mid_results
    

0 Geneformer-V2-104M_continue
    X_geneformer
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
1 Geneformer-V2-104M_CLcancer
    X_geneformer
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
2 hvg
    X_hvg
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
3 scvi_donor_id
    X_scVI
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
4 Geneformer-V2-316M
    X_geneformer
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
5 Geneformer-V2-104M
    X_geneformer
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
6 scgpt_cancer
    X_scGPT
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
7 scfoundation
    X_scfoundation
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
8 scgpt
    X_scGPT
louvain
louvain
louvain
louvain
louvain
louvain
louvain
louvain
lo

In [59]:
results_mean_df = pd.concat(results_mean, axis=1).T.set_index('model')

In [60]:
ind = results_mean_df.index.str.contains('__scvi')
results_mean_df = results_mean_df[~ind]
ind = results_mean_df.index =='scvi'
results_mean_df  = results_mean_df[~ind]
results_mean_df

,NMI_cluster/label,ARI_cluster/label,ASW_label,graph_conn,ASW_batch,ASW_label/batch,PCR_batch,avg_bio
model,,,,,,,,
Geneformer-V2-104M_continue,0.132666,0.096671,0.447959,0.65562,0.418344,0.745017,0.026559,0.225765
Geneformer-V2-104M_CLcancer,0.648102,0.506496,0.509267,0.969681,0.438935,0.840867,0.174468,0.554622
hvg,0.535608,0.26532,0.508813,0.965808,0.481053,0.950077,0.221971,0.436581
scvi_donor_id,0.675993,0.475049,0.593261,0.991059,0.479106,0.824882,0.181404,0.581434
Geneformer-V2-316M,0.712597,0.628071,0.507252,0.958874,0.452682,0.856805,0.178611,0.615973
Geneformer-V2-104M,0.680612,0.60996,0.503083,0.958886,0.437905,0.818556,0.165335,0.597885
scgpt_cancer,0.754777,0.731011,0.562042,0.980313,0.440433,0.862721,0.206015,0.68261
scfoundation,0.697104,0.594448,0.571232,0.988382,0.467406,0.860919,0.17787,0.620928
scgpt,0.727857,0.678384,0.562734,0.969631,0.44041,0.833574,0.161695,0.656325


In [47]:
results_std_df = pd.concat(results_std, axis=1).T.set_index('model')

In [61]:
ind = results_std_df.index.str.contains('__scvi')
results_std_df = results_std_df[~ind]
ind = results_std_df.index =='scvi'
results_std_df  = results_std_df[~ind]
results_std_df



,NMI_cluster/label,ARI_cluster/label,ASW_label,graph_conn,ASW_batch,ASW_label/batch,PCR_batch,avg_bio
model,,,,,,,,
Geneformer-V2-104M_continue,0.004502,0.007724,0.003895,0.011206,0.003627,0.013863,0.0041,0.004272
Geneformer-V2-104M_CLcancer,0.033129,0.09501,0.000919,0.01318,0.001869,0.008621,0.003719,0.042146
hvg,0.008983,0.019868,0.000636,0.003437,0.001728,0.002268,0.002147,0.006704
scvi_donor_id,0.013387,0.064733,0.001407,0.003579,0.000711,0.011621,0.002691,0.025956
Geneformer-V2-316M,0.026707,0.061946,0.001508,0.027473,0.001636,0.009403,0.003372,0.029362
Geneformer-V2-104M,0.033848,0.07044,0.002819,0.019316,0.00136,0.009964,0.003127,0.03461
scgpt_cancer,0.021461,0.055752,0.00158,0.009054,0.001594,0.007687,0.004552,0.025245
scfoundation,0.031704,0.075962,0.000937,0.005535,0.001236,0.010823,0.003262,0.035731
scgpt,0.012688,0.024508,0.00599,0.017241,0.001581,0.012625,0.002272,0.011714


In [62]:
results_std_df.to_csv('./fig2_metrics/metrics_10samples_std.csv')

In [63]:
results_mean_df.to_csv('./fig2_metrics/metrics_10samples_mean.csv')

In [68]:
# Convert to DataFrame with method as additional column
df_list = []
for method, records in all_results.items():
    df = pd.DataFrame(records)
    df["method"] = method
    df_list.append(df)

result_df = pd.concat(df_list, ignore_index=True)
result_df.head()

,NMI_cluster/label,ARI_cluster/label,ASW_label,graph_conn,ASW_batch,ASW_label/batch,PCR_batch,avg_bio,method
0,0.129519,0.097692,0.450335,0.650714,0.419690,0.722900,0.027858,0.225849,Geneformer-V2-104M_continue
1,0.135984,0.102432,0.448726,0.664223,0.416998,0.747465,0.033049,0.229047,Geneformer-V2-104M_continue
2,0.130433,0.087637,0.452150,0.644008,0.414649,0.759104,0.031897,0.223407,Geneformer-V2-104M_continue
3,0.127309,0.093911,0.446396,0.644266,0.419008,0.739000,0.024874,0.222539,Geneformer-V2-104M_continue
4,0.134912,0.098717,0.449302,0.645478,0.412620,0.737268,0.025222,0.227644,Geneformer-V2-104M_continue


In [69]:
ind = result_df.method.str.contains('__scvi')
result_df = result_df[~ind]
ind = result_df.method =='scvi'
result_df  = result_df[~ind]
result_df



,NMI_cluster/label,ARI_cluster/label,ASW_label,graph_conn,ASW_batch,ASW_label/batch,PCR_batch,avg_bio,method
0,0.129519,0.097692,0.450335,0.650714,0.419690,0.722900,0.027858,0.225849,Geneformer-V2-104M_continue
1,0.135984,0.102432,0.448726,0.664223,0.416998,0.747465,0.033049,0.229047,Geneformer-V2-104M_continue
2,0.130433,0.087637,0.452150,0.644008,0.414649,0.759104,0.031897,0.223407,Geneformer-V2-104M_continue
3,0.127309,0.093911,0.446396,0.644266,0.419008,0.739000,0.024874,0.222539,Geneformer-V2-104M_continue
4,0.134912,0.098717,0.449302,0.645478,0.412620,0.737268,0.025222,0.227644,Geneformer-V2-104M_continue
...,...,...,...,...,...,...,...,...,...
155,0.710987,0.663331,0.499808,0.973431,0.471967,0.870298,0.147988,0.624709,gf-6L-30M-i2048_continue
156,0.757802,0.706097,0.504240,0.967576,0.470649,0.884298,0.150742,0.656046,gf-6L-30M-i2048_continue
157,0.723683,0.674200,0.516433,0.964077,0.470683,0.877303,0.156095,0.638105,gf-6L-30M-i2048_continue
158,0.753129,0.699900,0.502409,0.977541,0.472319,0.872507,0.145492,0.651813,gf-6L-30M-i2048_continue


In [70]:
def map_groups(exp):

    if 'gf' in exp:
        return 'geneformer'
    elif 'Geneformer' in exp:
        return 'geneformer'
    elif 'scfoundation' in exp:
        return 'scFoundation'
    elif 'scimiarity' in exp:
        return 'SCimilarity'
    elif 'scgpt' in exp:
        return 'scgpt'
    elif any(x in exp for x in ['hvg', 'pca', 'scvi']):
        return 'baseline'
    else:
        return 'other'  # optional fallback

In [71]:
result_df['group'] = result_df['method'].map(map_groups)
result_df.set_index('method', inplace=True)
result_df.to_csv('./fig2_metrics/metrics_10_runs.csv')

In [50]:
# combined_results['group'] = combined_results['method'].map(map_groups)
# combined_results.set_index('method', inplace=True)
# combined_results.to_csv('./metrics/metrics_10_runs.csv')